<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day02-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 2 lab: working like a computational biologist {.unnumbered}

This lab continues from the Day 2 page's mini challenge, with less help
at each step:

- **Part A, fill in:** complete marked lines in code that otherwise
  works: explore the Becares RNA-seq table, plot it, test a relationship,
  and read a sequence from NCBI.
- **Part B, adapt:** change working code to answer a new question, and
  fix a cell with three bugs.
- **Part C, solve:** a small analysis of your own, from records to a
  table, a number, a plot and a written explanation.

Every `todo("...")` call marks code for you to write: replace the whole
`todo(...)` call with your code. Work through the cells **in order, top
to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

Parts A and B fetch records from NCBI, so you need an internet
connection. Run the notebook on the course server or in Colab (see the
Setup page). Answer the Day 2 lab quiz on Canvas with **your own
notebook's output**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

Entrez.email = "your.name@student.su.se"    # replace with your own address: NCBI requires one

URL = "https://raw.githubusercontent.com/JoachimGoedhart/VolcaNoseR/master/Becares-diffgenes_HFHC.csv"

def todo(what):
    '''Placeholder for code you write: replace the whole todo(...) call with your own code.'''
    raise NotImplementedError(f"TODO in this cell: {what}")

# Part A: Fill in

## A1. Load and inspect the table

The same dataset as on the Day 2 page: mouse liver RNA-seq,
high-fat/high-cholesterol diet against control (Becares et al. 2019).
This time, look at the column `lfcSE`: the **standard error** of each
log2 fold change, i.e. how uncertain the estimate is.

In [ ]:
df = pd.read_csv(URL)
print(df.shape)
print(df.head())

median_se = todo('the median of the lfcSE column')
max_padj = todo('the largest adjusted p-value (column padj) in the file')
print(f"median lfcSE:          {median_se:.3f}")
print(f"largest adjusted p:    {max_padj:.3f}")

## A2. Filter

A stricter threshold than the page's: adjusted p-value below 0.01.

In [ ]:
strict = todo('the rows of df with padj below 0.01')
n_up = todo('how many of those genes have a positive log2_FoldChange')
print(f"genes with padj < 0.01: {len(strict)}  (up: {n_up}, down: {len(strict) - n_up})")

## A3. Plot first, test second

Is the uncertainty of a fold change (`lfcSE`) related to how highly the
gene is expressed (`baseMean`)? Plot it first, with expression on a log
axis; then test.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
todo("scatter plot of lfcSE (y) against baseMean (x), small points")
ax.set_xscale("log")
ax.set_xlabel("mean expression (log scale)"); ax.set_ylabel("lfcSE (standard error of log2 FC)")
plt.tight_layout(); plt.show()

r_raw, p_raw = stats.pearsonr(df["baseMean"], df["lfcSE"])
print(f"Pearson, raw baseMean:  r = {r_raw:.2f}, p = {p_raw:.2g}")
rho, p_rho = todo('Spearman correlation between baseMean and lfcSE')
print(f"Spearman:               rho = {rho:.2f}, p = {p_rho:.2g}")

## A4. A mystery sequence

A 30-nucleotide fragment of a human gene. Which gene is revealed in A5.

In [ ]:
mystery = Seq("CGCAGGGAGCGGAGCGAGGAGCTGTCCGAG")

gc = todo('the GC fraction of mystery')
rev_comp = todo('the reverse complement of mystery')
protein = todo('mystery translated to protein')
print("GC fraction:        ", round(gc, 2))
print("reverse complement: ", rev_comp)
print("translated:         ", protein)

## A5. From NCBI to a local file and back

Fetch the full record for accession `NM_134268.5`, a human
globin-family gene, inside a `with` block. Then save it as a local FASTA
file and read it back, again with `with`.

In [ ]:
with todo("open an Entrez.efetch handle: database nuccore, id NM_134268.5, rettype gb, retmode text") as handle:
    record = SeqIO.read(handle, "genbank")
print("description:", record.description)
print("length (nt):", len(record.seq))

with open("mystery_gene.fasta", "w") as out:
    SeqIO.write(record, out, "fasta")
with open("mystery_gene.fasta") as handle:
    local = todo('read the FASTA file back as one record (SeqIO.read)')
print("read back from the local file:", local.id, len(local.seq))

position = todo('the 0-based position of the mystery fragment in the mRNA (str.find)')
print("the mystery fragment starts at position", position)

# Part B: Adapt

## B1. Adapt a query

The Day 2 page found the most significant genes with
`df.sort_values("padj").head()`. Adapt that idea: among the genes with
padj < 0.01 (your table `strict` from A2), which gene went **down** the
most?

In [ ]:
print(df.sort_values("padj").head()[["Gene", "log2_FoldChange", "padj"]])   # the page's query, for reference

most_down = todo('the row of strict with the most negative log2_FoldChange')
print(f"\nmost down-regulated gene with padj < 0.01: {most_down['Gene']} "
      f"(log2 FC {most_down['log2_FoldChange']:.2f})")

## B2. Fix three bugs

The cell below should compute two things:

1. the average GC fraction of three sequence fragments, weighting each
   fragment by its length, and
2. the number of genes in the table with a log2 fold change above 1.

It contains three bugs. One stops the cell with an `ImportError`, one
with a `KeyError`, and one lets the cell run but gives a wrong answer.
Use the routine from the Day 2 page (read the error, inspect,
documentation, verify) to fix all three. **Check that your answer is
plausible:** a weighted average must lie between the smallest and the
largest of the values it averages.

In [ ]:
from Bio.SeqUtils import GC

fragments = ["ATGGTGCATCTGACTCCTGAG", "CGCAGGGAGCGGAGCGAGGAGCTG", "ATGGTGCTGTCTCCTGCCGACAAGACC"]
gc_values = [GC(f) for f in fragments]
lengths = [len(f) for f in fragments]
weights = lengths * 2                  # weight each fragment by (twice) its length
weighted_gc = sum(g * w for g, w in zip(gc_values, weights)) / sum(weights)
print("GC per fragment:     ", [round(g, 3) for g in gc_values])
print("length-weighted GC:  ", round(weighted_gc, 3))

n_big = (df["log2FoldChange"] > 1).sum()
print("genes with log2 FC > 1:", n_big)

## B3. Adapt the mini challenge to new genes

The Day 2 page analysed six globins. The function below is the page's
mini-challenge code, packaged so that it works for any set of
accessions. Adapt it in two ways:

1. add a column `untranslated`: the mRNA length minus the length of the
   coding sequence (`len(cds.extract(record.seq))`);
2. use it on six more human globins from the alpha- and beta-globin
   gene clusters:

| Gene | Protein | RefSeq accession |
|---|---|---|
| *HBG1* | gamma-globin 1 (fetal) | `NM_000559.3` |
| *HBG2* | gamma-globin 2 (fetal) | `NM_000184.3` |
| *HBD* | delta-globin | `NM_000519.4` |
| *HBE1* | epsilon-globin (embryonic) | `NM_005330.4` |
| *HBZ* | zeta-globin (embryonic) | `NM_005332.3` |
| *HBQ1* | theta-globin | `NM_005331.5` |

In [ ]:
def globin_table(accessions):
    '''One row per gene: fetch all records in ONE request, then read length, GC and protein length.'''
    with Entrez.efetch(db="nuccore", id=",".join(accessions.values()), rettype="gb", retmode="text") as handle:
        records = list(SeqIO.parse(handle, "genbank"))
    rows = []
    for gene, record in zip(accessions, records):
        cds = [f for f in record.features if f.type == "CDS"][0]
        row = {"gene": gene, "accession": record.id, "mrna_length": len(record.seq),
               "gc": round(gc_fraction(record.seq), 3),
               "protein_length": len(cds.qualifiers["translation"][0])}
        row["untranslated"] = todo('mRNA length minus the length of the CDS')
        rows.append(row)
    table = pd.DataFrame(rows)
    assert list(table["accession"]) == list(accessions.values()), "records came back in a different order"
    return table

new_accessions = {"HBG1": "NM_000559.3", "HBG2": "NM_000184.3", "HBD": "NM_000519.4",
                  "HBE1": "NM_005330.4", "HBZ": "NM_005332.3", "HBQ1": "NM_005331.5"}  # TODO: a dict gene -> accession for the six genes in the table above
new_globins = globin_table(new_accessions)
new_globins

# Part C: Solve

## C1. Twelve globins: does a longer mRNA mean a longer protein?

Now without step-by-step help. Combine the page's six globins (*HBB*
`NM_000518.5`, *HBA1* `NM_000558.5`, *HBA2* `NM_000517.6`, *MB*
`NM_001382809.1`, *NGB* `NM_021257.4`, *CYGB* `NM_134268.5`) with your
six new ones, and:

1. build one DataFrame, `all_globins`, with twelve rows and the columns
   from B3, plus a column `group` ("page" or "new") and a column
   `coding_fraction` (the fraction of the mRNA that is coding sequence);
2. compute the Pearson correlation between mRNA length and protein
   length over all twelve genes;
3. plot protein length against mRNA length, one colour per group, with
   each point labelled by its gene;
4. check how much the correlation depends on a single gene: recompute it
   without the gene with the longest protein;
5. write two or three sentences, in the markdown cell below the plot,
   explaining what the plot shows about mRNAs and proteins.

Hints: `globin_table` does most of the work; `pd.concat([a, b])` stacks
two DataFrames; seaborn's `hue=` colours by a column; `ax.annotate`
labels a point.

In [ ]:
page_accessions = todo("a dict gene -> accession for the page's six globins")
all_globins = todo("one DataFrame: both tables stacked, with the columns group and coding_fraction added")
r, p = todo("Pearson correlation between mrna_length and protein_length over all twelve genes")
r_no, p_no = todo("the same correlation without the gene with the longest protein")
# then: the plot, one colour per group, each point labelled with its gene

**Your explanation (two or three sentences):** *replace this line with
what the plot shows about the relationship between mRNA length and
protein length, and why.*

## Done

When every cell runs, answer the Day 2 lab quiz on Canvas with your own
output, and paste your explanation from C1 into the last question.